# Lesson 20 — Queue、Event 与 race condition — Solution

这是独立答案 Notebook。先完成主课再打开它。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

In [ ]:
%%academy_lab 20-sync-and-race solution exercise pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& device) : device_(device) {
        }
        void tick();

    private:
        Device& device_;
        std::uint32_t observed_pulses_ = 0;
        int pending_ = 0;
    };
}

namespace academy {
    void Controller::tick() {
        if (device_.input.pulses > observed_pulses_)
            pending_ += static_cast<int>(device_.input.pulses - observed_pulses_);
        observed_pulses_ = device_.input.pulses;
        device_.output.error = device_.input.fault;
        if (device_.input.fault)
            device_.output.state = "fault";
        else if (!device_.input.connected)
            device_.output.state = "offline";
        else if (pending_ > 0) {
            --pending_;
            ++device_.output.events;
            device_.output.state = "processing";
        } else
            device_.output.state = "waiting";
        device_.output.queue_depth = pending_;
    }
}
